[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t04_waves.ipynb)

# Turning becomes a wave: day length in Delhi, London and Singapore

A year is one turn of the Earth around the Sun, so the length of a day rises and falls like a wave. In London it
swings from 16 h 38 m in June to 7 h 49 m in December. In New Delhi, from 13 h 58 m to 10 h 19 m. In Singapore it
barely moves: 12 h 12 m and 12 h 03 m. Sydney is half a year out of step.

This notebook unrolls the **turning arrow** in time, and every trig graph appears.

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Unroll the turn

On the left, the arrow turns. On the right, we plot the height of its tip (its vertical projection) against how far it has
turned. As the arrow goes round, the dot draws a wave. That wave is the **graph of sin x**.

In [ ]:
def unroll(angle_deg=130.0):
    a = math.radians(angle_deg)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 2.4]})
    circle_axes(ax, lim=1.25, labels=False)
    arrow(ax, a); ax.plot([math.cos(a), 1.25], [math.sin(a)] * 2, color=CORAL, ls=":", lw=1.2)
    ax.set_title(f"{angle_deg:.0f}°")
    xs = np.linspace(0, 2 * np.pi, 400)
    bx.plot(xs, np.sin(xs), color=DIM, lw=1.2)
    done = xs[xs <= a]
    bx.plot(done, np.sin(done), color=SAGE, lw=3.2)
    bx.plot([a], [math.sin(a)], "o", color=CORAL, ms=10)
    bx.set_ylim(-1.25, 1.25); pi_ticks(bx, 0, 2 * np.pi); bx.set_title("the height of the tip, unrolled: sin x")
    plt.tight_layout(); plt.show()

interact(unroll, angle_deg=FloatSlider(value=130, min=0, max=360, step=5, description="angle (°)"));

**Notice.** One full turn draws one full wave. Keep turning and the wave repeats: the **period** of sin x is 2π.

## 2 · Cosine is the same wave, a quarter turn earlier

The horizontal projection does the same thing, just a quarter of a turn ahead: cos x = sin(x + π/2). Slide the sine wave to
the left and watch it land on the cosine.

In [ ]:
def shift(s_deg=0.0):
    s = math.radians(s_deg)
    x = np.linspace(-2 * np.pi, 2 * np.pi, 800)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(x, np.cos(x), color=TEAL, lw=6, alpha=0.5, label="cos x")
    ax.plot(x, np.sin(x + s), color=CORAL, label=f"sin(x + {s_deg:.0f}°)")
    ax.set_ylim(-1.3, 1.3); pi_ticks(ax); ax.legend(loc="lower right")
    ax.set_title("they match when the shift is 90°, a quarter turn")
    plt.show()

interact(shift, s_deg=FloatSlider(value=0, min=0, max=180, step=5, description="shift (°)"));

## 3 · Day length is a turning-arrow wave

These day lengths come from the US National Oceanic and Atmospheric Administration's solar equations, which turn a
date and a latitude into the Sun's position. They count sunrise and sunset as NOAA does (when the top of the Sun
meets the horizon). The formulas are in the next box, so you can see every step.

In [ ]:
import datetime as dt

def day_length(lat, lon, doy, year=2026):
    """Hours of daylight on day `doy` of `year`, from NOAA's solar calculator equations
    (gml.noaa.gov/grad/solcalc/calcdetails.html), at local solar noon."""
    d0 = dt.date(year, 1, 1).toordinal() - dt.date(2000, 1, 1).toordinal()
    jd = 2451544.5 + d0 + (doy - 1) + (12 - lon / 15) / 24          # Julian day
    T = (jd - 2451545.0) / 36525                                    # centuries since 2000
    L0 = (280.46646 + T * (36000.76983 + T * 0.0003032)) % 360       # the Sun's mean longitude
    M = np.radians(357.52911 + T * (35999.05029 - 0.0001537 * T))    # its mean anomaly
    C = (np.sin(M) * (1.914602 - T * (0.004817 + 0.000014 * T)) + np.sin(2 * M) * (0.019993 - 0.000101 * T)
         + np.sin(3 * M) * 0.000289)                                # equation of centre
    om = np.radians(125.04 - 1934.136 * T)
    lam = np.radians(L0 + C - 0.00569 - 0.00478 * np.sin(om))        # apparent longitude
    eps0 = 23 + (26 + (21.448 - T * (46.815 + T * (0.00059 - T * 0.001813))) / 60) / 60
    eps = np.radians(eps0 + 0.00256 * np.cos(om))                    # tilt of the Earth's axis
    dec = np.arcsin(np.sin(eps) * np.sin(lam))                       # the Sun's declination
    la = np.radians(lat)
    cosH = np.cos(np.radians(90.833)) / (np.cos(la) * np.cos(dec)) - np.tan(la) * np.tan(dec)
    return 8 * np.degrees(np.arccos(np.clip(cosH, -1, 1))) / 60       # hours of daylight

CITIES = {"New Delhi": (28.6139, 77.2090, TEAL), "London": (51.5074, -0.1278, CORAL),
          "Singapore": (1.3521, 103.8198, SAGE), "Sydney": (-33.8688, 151.2093, LILAC)}
hm = lambda h: f"{int(round(h * 60)) // 60} h {int(round(h * 60)) % 60:02d} m"
print(f"{'':10} {'21 June':>10} {'21 December':>13}")
for name, (lat, lon, _) in CITIES.items():
    print(f"{name:10} {hm(day_length(lat, lon, 172)):>10} {hm(day_length(lat, lon, 355)):>13}")
assert abs(day_length(51.5074, -0.1278, 172) - (16 + 38 / 60)) < 1.5 / 60      # within a minute or so of NOAA's tables

In [ ]:
def day_lengths(highlight="London"):
    days = np.arange(1, 366)
    fig, ax = plt.subplots(figsize=(11.5, 4.8))
    for name, (lat, lon, col) in CITIES.items():
        h = day_length(lat, lon, days)
        ax.plot(days, h, color=col, lw=4 if name == highlight else 1.8, alpha=1 if name == highlight else 0.7, label=name)
    ax.axhline(12, color=DIM, ls="--", lw=1)
    ax.set_xticks([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335])
    ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
    ax.set_ylabel("hours of daylight"); ax.set_ylim(6, 18); ax.legend(loc="upper right", ncol=2)
    ax.set_title("day length through 2026: one turn of the Earth around the Sun")
    plt.show()

interact(day_lengths, highlight=Dropdown(options=list(CITIES), value="London", description="city"));

**Notice.** All four curves share the same **period**, one year. Their **amplitudes** differ: big in London, tiny in
Singapore, near the equator. Sydney's curve is upside down compared with London's: a **phase shift** of half a year.
The curves are close to sine waves, though not exactly.

## 4 · Amplitude, period and shift (beyond the syllabus, but useful)

Any wave like these can be written as **A sin(B x + C)**: A stretches it up and down (amplitude), B squeezes it
sideways (the period becomes 2π ÷ B), and C slides it (the shift).

In [ ]:
def explorer(A=1.0, B=1.0, C=0.0):
    x = np.linspace(-2 * np.pi, 2 * np.pi, 1000)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(x, np.sin(x), color=DIM, lw=1.2, label="sin x")
    ax.plot(x, A * np.sin(B * x + C), color=CORAL, label=f"{A:.1f} sin({B:.1f}x + {C:.2f})")
    ax.set_ylim(-3.2, 3.2); pi_ticks(ax); ax.legend(loc="lower right")
    ax.set_title(f"amplitude {abs(A):.1f},  period 2π ÷ {B:.1f} = {2 * np.pi / B:.2f}")
    plt.show()

interact(explorer, A=FloatSlider(value=1, min=0.2, max=3, step=0.1), B=FloatSlider(value=1, min=0.5, max=4, step=0.1),
         C=FloatSlider(value=0, min=-math.pi, max=math.pi, step=0.05));

## 5 · All six graphs

tan, cot, sec and cosec are ratios of the two projections, so their graphs come from sin and cos. Each breaks where its
bottom is zero: those **gaps** (dashed lines) are outside its domain. tan and cot repeat every π; the other four
every 2π.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 3000)
graphs = [("sin x", np.sin(x), [], SAGE), ("cos x", np.cos(x), [], TEAL),
          ("tan x", np.tan(x), np.arange(-1.5, 2, 1) * np.pi, CORAL), ("cot x", 1 / np.tan(x), np.arange(-2, 3) * np.pi, LILAC),
          ("sec x", 1 / np.cos(x), np.arange(-1.5, 2, 1) * np.pi, TEAL), ("cosec x", 1 / np.sin(x), np.arange(-2, 3) * np.pi, SAGE)]
fig, axs = plt.subplots(3, 2, figsize=(12, 10))
for ax, (name, y, poles, col) in zip(axs.flat, graphs):
    ax.plot(x, gaps(y, 5), color=col)
    for p in poles: ax.axvline(p, color=DIM, ls="--", lw=0.9)
    ax.set_ylim(-4, 4); pi_ticks(ax); ax.set_title(name)
plt.tight_layout(); plt.show()

**Notice.** sin and cos never leave [−1, 1]. tan and cot take every value. sec and cosec never come between −1 and 1:
their graphs bounce off those lines.

## 6 · The identity gallery

**cos x = sin(x + π/2).** The horizontal projection is the vertical one, a quarter turn earlier.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
both_sides(ax, x, np.cos(x), np.sin(x + np.pi / 2), "cos x  and  sin(x + π/2)"); ax.legend(loc="lower right"); plt.show()
check_identity(np.cos, lambda x: np.sin(x + np.pi / 2));

**sin(x + 2π) = sin x**, **cos(x + 2π) = cos x**, **tan(x + π) = tan x.** A full turn repeats sin and cos; half a turn
already repeats tan.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.4))
both_sides(axs[0], x, np.sin(x + 2 * np.pi), np.sin(x), "sin(x + 2π)  and  sin x")
both_sides(axs[1], x, np.tan(x + np.pi), np.tan(x), "tan(x + π)  and  tan x", ylim=(-5, 5))
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(lambda x: np.sin(x + 2 * np.pi), np.sin)
check_identity(lambda x: np.cos(x + 2 * np.pi), np.cos)
check_identity(lambda x: np.tan(x + np.pi), np.tan, ok=away_from_cos0);

**sin(−x) = −sin x** (an odd function: its graph turns onto itself under a half turn about the origin) and
**cos(−x) = cos x** (an even function: its graph is its own mirror image in the y axis).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.4))
both_sides(axs[0], x, np.sin(-x), -np.sin(x), "sin(−x)  and  −sin x")
both_sides(axs[1], x, np.cos(-x), np.cos(x), "cos(−x)  and  cos x")
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(lambda x: np.sin(-x), lambda x: -np.sin(x))
check_identity(lambda x: np.cos(-x), np.cos);

**sec(x + 2π) = sec x** and **cosec(x + 2π) = cosec x**, because they are 1 ÷ cos and 1 ÷ sin.

In [ ]:
check_identity(lambda x: 1 / np.cos(x + 2 * np.pi), lambda x: 1 / np.cos(x), ok=away_from_cos0)
check_identity(lambda x: 1 / np.sin(x + 2 * np.pi), lambda x: 1 / np.sin(x), ok=away_from_sin0);

## 7 · On paper: the CBSE way

**Example 1.** State the domain and range of tan x.

1. tan x = sin x ÷ cos x, so it is undefined where cos x = 0: at odd multiples of π/2.
2. **Domain:** all real x except x = (2n + 1) π/2, for any whole number n.
3. Near each gap the graph runs off to + and − infinity, so **range:** all real numbers.

In [ ]:
print([f"{math.tan(v):.1f}" for v in (math.pi / 2 - 0.1, math.pi / 2 - 0.01, math.pi / 2 + 0.01)])

**Example 2.** Which is greater, sin 1 or sin 2? (The angles are in radians.)

1. 1 and 2 are both between 0 and π, where sin is positive.
2. The sine graph peaks at π/2 ≈ 1.57. The point 1 is 0.57 to its left; the point 2 is 0.43 to its right.
3. The wave is symmetric about its peak, so the closer point is higher: **sin 2 is greater**.

In [ ]:
print(f"sin 1 = {math.sin(1):.4f},  sin 2 = {math.sin(2):.4f}")

**Try it.**
1. For x between 0 and 2π, where is sin x positive and cos x negative?
2. What is the range of cos x for x between 0 and π?

<details><summary>Answers</summary>

1. Between π/2 and π: the second quarter.
2. From −1 to 1: the horizontal projection goes all the way from right to left.

</details>

## Check yourself
1. What is the period of tan x?
2. What is the amplitude of 3 sin x?
3. Which city's day length hardly changes through the year, and why?

<details><summary>Answers</summary>

1. π.
2. 3.
3. Singapore: it sits almost on the equator, where day and night stay close to twelve hours each.

</details>

## Sources
NOAA Global Monitoring Laboratory, Solar Calculator and its calculation details (gml.noaa.gov/grad/solcalc). NCERT
class 11, chapter 3 (graphs, domain and range). CBSE Mathematics syllabus 2026–27. All retrieved 2026-10-21.